# Проверка качества оценщика

**Что проверяем.** Оценщик (`app/agents/evaluator.py`) ставит каждому ответу баллы от 1 до 5 по четырём критериям:

- **Конкретика**: есть ли реальный пример, цифры, инструменты, результат;
- **Структура**: понятно ли построен ответ (ситуация, задача, действия, результат);
- **Релевантность**: отвечает ли человек именно на заданный вопрос;
- **Глубина**: видно ли понимание: почему так, какие были варианты, какой вывод.

Критерии взяты из того, как оценивают ответы на реальных собеседованиях: интервьюеры ждут ответ по схеме STAR (ситуация, задача, действия, результат) и смотрят, есть ли конкретный пример, отвечает ли кандидат на вопрос и понимает ли он, что делает.

**Тестовые данные** лежат в `eval/cases.json`: 8 вакансий разных профессий, в каждой 2 вопроса, и на каждый вопрос есть пара ответов: *хороший* (конкретный, с примером и результатом) и *плохой* (расплывчатый, общие слова).

**Метрика.** Для каждой пары оцениваем оба ответа отдельно и складываем 4 балла (сумма от 4 до 20).
Пара считается **победой**, если сумма у хорошего ответа больше, чем у плохого; **ничьей** при равенстве; **поражением**, если больше у плохого.

**Главная метрика = доля побед = число побед / число пар.** Ничьи победой не считаются. Случайный оценщик дал бы около 50%, идеальный 100%.

Вызовы идут строго по очереди: бесплатный тариф GigaChat принимает один запрос за раз. Ключ берётся из `.env` в корне проекта.

In [1]:
import json
import os
import re
import sys
import time
from pathlib import Path

# Ноутбук можно открыть из папки eval/ или из корня проекта.
ROOT = Path.cwd().parent if Path.cwd().name == "eval" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd

from app.agents.evaluator import CRITERIA, FAILED_COMMENT, score_answer
from app.config import settings

CASES_PATH = ROOT / "eval" / "cases.json"
RESULTS_PATH = ROOT / "eval" / "results.csv"
README_PATH = ROOT / "eval" / "README.md"

cases = json.loads(CASES_PATH.read_text(encoding="utf-8"))
pair_count = sum(len(case["questions"]) for case in cases)
print(f"Вакансий: {len(cases)}, пар ответов: {pair_count}, вызовов оценщика: {pair_count * 2}")
print(f"Модель: {settings.model}")

Вакансий: 8, пар ответов: 16, вызовов оценщика: 32
Модель: GigaChat-3-Pro


## Прогон оценщика

`RERUN = True` заново вызывает GigaChat для всех ответов (32 вызова, несколько минут). `RERUN = False` берёт сохранённые результаты из `eval/results.csv` и не тратит токены.

In [2]:
RERUN = True

if RERUN or not RESULTS_PATH.exists():
    rows = []
    total_calls = pair_count * 2
    for case in cases:
        for q_number, item in enumerate(case["questions"], start=1):
            for kind in ("good", "bad"):
                started = time.time()
                result = score_answer(item["question"], item[kind])  # вызовы строго по очереди
                seconds = time.time() - started
                row = {"case_id": case["id"], "profession": case["profession"], "q": q_number, "kind": kind}
                row.update({name: result.scores.get(name) for name in CRITERIA})
                row["total"] = sum(result.scores.get(name, 0) for name in CRITERIA)
                row["followup"] = result.needs_followup
                row["failed"] = result.comment == FAILED_COMMENT
                row["comment"] = result.comment
                row["seconds"] = round(seconds, 1)
                rows.append(row)
                print(f"[{len(rows)}/{total_calls}] {case['id']} вопрос {q_number} {kind}: "
                      f"сумма {row['total']} ({seconds:.1f} с)")
    results = pd.DataFrame(rows)
    results.to_csv(RESULTS_PATH, index=False, encoding="utf-8")
else:
    results = pd.read_csv(RESULTS_PATH)

failures = int(results["failed"].sum())
if failures:
    print(f"ВНИМАНИЕ: оценщик не смог оценить {failures} ответов, у них стоят баллы по умолчанию.")
results[["case_id", "q", "kind", *CRITERIA, "total", "followup", "comment"]]

[1/32] python_backend вопрос 1 good: сумма 20 (1.7 с)


[2/32] python_backend вопрос 1 bad: сумма 7 (0.8 с)


[3/32] python_backend вопрос 2 good: сумма 19 (1.0 с)


[4/32] python_backend вопрос 2 bad: сумма 7 (0.8 с)


[5/32] data_analyst вопрос 1 good: сумма 19 (1.0 с)


...

[29/32] ux_designer вопрос 1 good: сумма 19 (1.1 с)


[30/32] ux_designer вопрос 1 bad: сумма 7 (2.4 с)


[31/32] ux_designer вопрос 2 good: сумма 19 (1.2 с)


[32/32] ux_designer вопрос 2 bad: сумма 7 (0.9 с)


,case_id,q,kind,Конкретика,Структура,Релевантность,Глубина,total,followup,comment
0,python_backend,1,good,5,5,5,5,20,False,Очень конкретный и структурированный ответ с п...
1,python_backend,1,bad,1,2,3,1,7,True,"Нет ни одного примера, нет описания конкретных..."
2,python_backend,2,good,5,5,5,4,19,False,Очень конкретный и структурированный ответ с п...
3,python_backend,2,bad,1,2,3,1,7,True,"Ответ слишком общий, нет ни одного примера инс..."
4,data_analyst,1,good,5,5,5,4,19,False,Очень конкретный и структурированный ответ с ц...
5,data_analyst,1,bad,1,2,3,1,7,True,"Нет ни одного примера, не указано, что именно ..."
6,data_analyst,2,good,5,5,5,5,20,False,"Ответ содержит конкретный пример, цифры, инстр..."
7,data_analyst,2,bad,1,2,3,1,7,True,"Ответ слишком общий, нет ни одного примера или..."
8,qa_engineer,1,good,5,5,5,4,19,False,"Подробный пример с цифрами, инструментами и ре..."
9,qa_engineer,1,bad,1,2,3,1,7,True,"Нет ни одного примера, описания сути бага или ..."


## Метрика: хороший ответ против плохого

In [3]:
keys = ["case_id", "profession", "q"]
columns = [*keys, *CRITERIA, "total", "followup"]
pairs = results[results["kind"] == "good"][columns].merge(
    results[results["kind"] == "bad"][columns], on=keys, suffixes=("_good", "_bad"))
pairs["margin"] = pairs["total_good"] - pairs["total_bad"]
pairs["outcome"] = pairs["margin"].apply(lambda m: "победа" if m > 0 else ("ничья" if m == 0 else "поражение"))

n = len(pairs)
wins = int((pairs["outcome"] == "победа").sum())
ties = int((pairs["outcome"] == "ничья").sum())
losses = int((pairs["outcome"] == "поражение").sum())
win_rate = wins / n
criterion_win_rate = {name: float((pairs[f"{name}_good"] > pairs[f"{name}_bad"]).mean()) for name in CRITERIA}
avg_good = pairs["total_good"].mean()
avg_bad = pairs["total_bad"].mean()
followup_good = pairs["followup_good"].mean()
followup_bad = pairs["followup_bad"].mean()

print(f"Пар: {n}")
print(f"ДОЛЯ ПОБЕД: {win_rate:.0%} ({wins} из {n}); ничьих: {ties}; поражений: {losses}")
print(f"Средняя сумма: хороший {avg_good:.1f}, плохой {avg_bad:.1f}, средний отрыв {pairs['margin'].mean():.1f}")
print("Доля пар, где хороший ответ выше по отдельному критерию:")
for name, value in criterion_win_rate.items():
    print(f"  {name}: {value:.0%}")
print(f"Уточнение запрошено: у хороших {followup_good:.0%}, у плохих {followup_bad:.0%}")
print(f"Сбоев оценщика: {failures}")
pairs[["profession", "q", "total_good", "total_bad", "margin", "outcome"]]

Пар: 16
ДОЛЯ ПОБЕД: 100% (16 из 16); ничьих: 0; поражений: 0
Средняя сумма: хороший 19.1, плохой 7.0, средний отрыв 12.1
Доля пар, где хороший ответ выше по отдельному критерию:
  Конкретика: 100%
  Структура: 100%
  Релевантность: 100%
  Глубина: 100%
Уточнение запрошено: у хороших 0%, у плохих 100%
Сбоев оценщика: 0


,profession,q,total_good,total_bad,margin,outcome
0,Python-разработчик (бэкенд),1,20,7,13,победа
1,Python-разработчик (бэкенд),2,19,7,12,победа
2,Аналитик данных,1,19,7,12,победа
3,Аналитик данных,2,20,7,13,победа
4,QA-инженер (тестировщик),1,19,7,12,победа
5,QA-инженер (тестировщик),2,19,7,12,победа
6,Продакт-менеджер,1,19,7,12,победа
7,Продакт-менеджер,2,19,7,12,победа
8,Менеджер по продажам B2B,1,19,7,12,победа
9,Менеджер по продажам B2B,2,19,7,12,победа


## Результаты по вакансиям

In [4]:
by_vacancy = pairs.groupby("profession", sort=False).agg(
    пар=("outcome", "size"),
    побед=("outcome", lambda s: int((s == "победа").sum())),
    средний_отрыв=("margin", "mean"),
).round(1)
by_vacancy

,пар,побед,средний_отрыв
profession,,,
Python-разработчик (бэкенд),2,2,12.5
Аналитик данных,2,2,12.5
QA-инженер (тестировщик),2,2,12.0
Продакт-менеджер,2,2,12.0
Менеджер по продажам B2B,2,2,12.0
HR-рекрутер,2,2,12.0
Бухгалтер,2,2,12.0
UX/UI-дизайнер,2,2,12.0


## Запись итога в eval/README.md

Ячейка заменяет блок между маркерами `<!-- RESULTS:START -->` и `<!-- RESULTS:END -->` в `eval/README.md`.

In [5]:
lines = [
    f"Последний прогон: {time.strftime('%Y-%m-%d')}, модель `{settings.model}`, пар: {n}.",
    "",
    f"- **Доля побед: {win_rate:.0%}** ({wins} из {n}); ничьих: {ties}; поражений: {losses}.",
    f"- Средняя сумма баллов: хороший ответ {avg_good:.1f}, плохой {avg_bad:.1f} (из 20).",
    "- Доля пар, где хороший ответ выше по отдельному критерию: "
    + ", ".join(f"{name} {value:.0%}" for name, value in criterion_win_rate.items()) + ".",
    f"- Уточняющий вопрос запрошен: у хороших ответов {followup_good:.0%}, у плохих {followup_bad:.0%}.",
    f"- Сбоев оценщика: {failures}.",
    "",
    "| Вакансия | Пар | Побед | Средний отрыв |",
    "|---|---|---|---|",
]
for profession, row in by_vacancy.iterrows():
    lines.append(f"| {profession} | {int(row['пар'])} | {int(row['побед'])} | {row['средний_отрыв']:.1f} |")

block = "<!-- RESULTS:START -->\n" + "\n".join(lines) + "\n<!-- RESULTS:END -->"
readme = README_PATH.read_text(encoding="utf-8")
readme = re.sub(r"<!-- RESULTS:START -->.*?<!-- RESULTS:END -->", lambda _: block, readme, flags=re.S)
README_PATH.write_text(readme, encoding="utf-8")
print("\n".join(lines))

Последний прогон: 2026-10-19, модель `GigaChat-3-Pro`, пар: 16.

- **Доля побед: 100%** (16 из 16); ничьих: 0; поражений: 0.
- Средняя сумма баллов: хороший ответ 19.1, плохой 7.0 (из 20).
- Доля пар, где хороший ответ выше по отдельному критерию: Конкретика 100%, Структура 100%, Релевантность 100%, Глубина 100%.
- Уточняющий вопрос запрошен: у хороших ответов 0%, у плохих 100%.
- Сбоев оценщика: 0.

| Вакансия | Пар | Побед | Средний отрыв |
|---|---|---|---|
| Python-разработчик (бэкенд) | 2 | 2 | 12.5 |
| Аналитик данных | 2 | 2 | 12.5 |
| QA-инженер (тестировщик) | 2 | 2 | 12.0 |
| Продакт-менеджер | 2 | 2 | 12.0 |
| Менеджер по продажам B2B | 2 | 2 | 12.0 |
| HR-рекрутер | 2 | 2 | 12.0 |
| Бухгалтер | 2 | 2 | 12.0 |
| UX/UI-дизайнер | 2 | 2 | 12.0 |


## Пример итогового разбора тренера

Собираем короткое интервью по первой вакансии: на первый вопрос даём хороший ответ, на второй плохой, и смотрим, что напишет тренер. Это один дополнительный вызов GigaChat. Здесь нет метрики, только чтобы глазами проверить, что разбор опирается на диалог и ничего не выдумывает.

В приложении тренер пишет четыре раздела. Здесь к промпту `prompts/coach.md` добавляется пятый, «Лучший и худший ответ», а в сообщение подставляются номера лучшего и худшего ответа по оценкам. Этот раздел нужен только для проверки и сравнения моделей, пользователь его не видит.

In [6]:
RUN_COACH = True

# Раздел только для проверки модели: в приложении его нет, пользователю он не нужен.
BEST_WORST_SECTION = (
    "### Лучший и худший ответ\n"
    "Назови номер и тему лучшего ответа и объясни в 1–2 предложениях, чем он хорош. "
    "Так же для худшего. Если в сообщении указано, какие ответы лучший и худший по оценкам, "
    "используй эти номера.\n\n"
)

if RUN_COACH:
    from app.agents.coach import final_report
    from app.prompts import load_prompt
    from app.schemas import AnswerScore, InterviewPlan, Turn

    coach_prompt = load_prompt("coach")
    # Если в prompts/coach.md поменяли заголовки, лучше упасть, чем молча собрать не тот промпт.
    assert "ровно эти четыре раздела" in coach_prompt
    assert "### Как можно было ответить лучше" in coach_prompt
    coach_prompt = (coach_prompt
                    .replace("ровно эти четыре раздела", "ровно эти пять разделов")
                    .replace("### Как можно было ответить лучше",
                             BEST_WORST_SECTION + "### Как можно было ответить лучше"))

    case = cases[0]
    plan = InterviewPlan(role=case["profession"], level="middle",
                         topics=[item["question"] for item in case["questions"]])
    history, scores = [], []
    for q_number, (item, kind) in enumerate(zip(case["questions"], ["good", "bad"]), start=1):
        history.append(Turn(question=item["question"], answer=item[kind]))
        row = results[(results["case_id"] == case["id"]) & (results["q"] == q_number) & (results["kind"] == kind)].iloc[0]
        scores.append(AnswerScore(scores={name: int(row[name]) for name in CRITERIA},
                                  comment=row["comment"] if isinstance(row["comment"], str) else "",
                                  needs_followup=bool(row["followup"])))
    print(final_report(plan, history, scores, system_prompt=coach_prompt))

### Что получилось хорошо

1. В ответе на первый вопрос вы привели конкретный пример из реального проекта: указали, какой эндпоинт был медленным и сколько времени занимал отклик.
2. Вы чётко описали шаги диагностики (включили логирование запросов) и выявили проблему — N+1 запросы через ORM.
3. Решение было структурировано по пунктам: оптимизация выборки с помощью select_related, создание индекса, пагинация.
4. Результат измерим: время ответа сократилось с 4 секунд до 180 мс, а также добавлен CI-тест для предотвращения регрессии.

### Что было слабо

1. Во втором вопросе не хватило конкретики: вы упомянули тестирование в целом, но не назвали ни одного инструмента или подхода.
2. Не описан сам процесс организации тестов: как они пишутся, где хранятся, кто их запускает, есть ли покрытие разных уровней (unit/integration).
3. Ответ слишком общий («иногда вручную, иногда автоматически»), без примеров ситуаций, когда выбирается тот или иной способ.
4. Нет упоминания о целях тестирования: каки